In [1]:
import torch
import torch.nn.functional as F
from aeon.datasets import load_from_ts_file
import numpy as np

In [ ]:
# Creates array of tensors
torch.arange(5)

tensor([0, 1, 2, 3, 4])

In [ ]:
# Creates a 1D tensor of 8 steps betweem 0 and 100
indices = torch.linspace(0, 100, 8)
indices

tensor([  0.0000,  14.2857,  28.5714,  42.8571,  57.1429,  71.4286,  85.7143,
        100.0000])

In [ ]:
# Stacks 2 tensors: (i_n, i_{n+1})
intervals_n = torch.stack((indices[:-1], indices[1:]), 1)
intervals_n

tensor([[  0.0000,  14.2857],
        [ 14.2857,  28.5714],
        [ 28.5714,  42.8571],
        [ 42.8571,  57.1429],
        [ 57.1429,  71.4286],
        [ 71.4286,  85.7143],
        [ 85.7143, 100.0000]])

## Testing Loading

In [2]:
base_path = "C:/Users/Gowtham/ROOTDIR/Career/Studies/PG - UCD/Learning/Autumn/AI for Time Series/group project/code/data/Car/Car"

X_train, y_train = load_from_ts_file(f"{base_path}_TRAIN.ts")
X_train.shape, y_train.shape

((60, 1, 577), (60,))

## Make input representations

In [19]:
X = torch.tensor(X_train[0], dtype=torch.float32).unsqueeze(1)

def calculate_input_representations(X):
    original = X
    first_diff = torch.diff(X, dim=-1)
    first_diff_padded = F.pad(first_diff, (2, 2), mode='replicate')
    first_diff_smoothed = F.avg_pool1d(first_diff_padded, kernel_size=5, stride=1)
    second_diff = torch.diff(X, n=2, dim=-1)
    series_fft = torch.fft.rfft(X, dim=-1).abs()
    
    return {
        "original": original,
        "first_diff_smoothed": first_diff_smoothed,
        "second_diff": second_diff,
        "fft": series_fft
    }

input_representations = calculate_input_representations(X)
for key, value in input_representations.items():
    print(f"Shape of {key}: {value.shape}")

Shape of original: torch.Size([1, 1, 577])
Shape of first_diff_smoothed: torch.Size([1, 1, 576])
Shape of second_diff: torch.Size([1, 1, 575])
Shape of fft: torch.Size([1, 1, 289])


## Make intervals

In [20]:
def make_intervals(input_length, depth):
    exponent = min(depth, int(np.log2(input_length)) + 1)
    intervals = []
    
    for dyad_step in 2 ** np.arange(exponent):
        indices = torch.linspace(0, input_length, dyad_step + 1).long()
        dyad_intervals = torch.stack((indices[:-1], indices[1:]), dim=1)
        intervals.append(dyad_intervals)
        
        if dyad_step > 1 and dyad_intervals.diff().median() > 1:
            shift = np.ceil(input_length / (dyad_step * 2))
            shifted_intervals = dyad_intervals[:-1] + shift
            intervals.append(shifted_intervals)
        print(f"At dyad_step: {dyad_step}, intervals: {intervals}")
            
    return torch.cat(intervals)
        
intervals = make_intervals(X_train.shape[-1], 6)
intervals

At dyad_step: 1, intervals: [tensor([[  0, 577]])]
At dyad_step: 2, intervals: [tensor([[  0, 577]]), tensor([[  0, 288],
        [288, 577]]), tensor([[145., 433.]])]
At dyad_step: 4, intervals: [tensor([[  0, 577]]), tensor([[  0, 288],
        [288, 577]]), tensor([[145., 433.]]), tensor([[  0, 144],
        [144, 288],
        [288, 432],
        [432, 577]]), tensor([[ 73., 217.],
        [217., 361.],
        [361., 505.]])]
At dyad_step: 8, intervals: [tensor([[  0, 577]]), tensor([[  0, 288],
        [288, 577]]), tensor([[145., 433.]]), tensor([[  0, 144],
        [144, 288],
        [288, 432],
        [432, 577]]), tensor([[ 73., 217.],
        [217., 361.],
        [361., 505.]]), tensor([[  0,  72],
        [ 72, 144],
        [144, 216],
        [216, 288],
        [288, 360],
        [360, 432],
        [432, 504],
        [504, 577]]), tensor([[ 37., 109.],
        [109., 181.],
        [181., 253.],
        [253., 325.],
        [325., 397.],
        [397., 469.],
    

tensor([[  0., 577.],
        [  0., 288.],
        [288., 577.],
        [145., 433.],
        [  0., 144.],
        [144., 288.],
        [288., 432.],
        [432., 577.],
        [ 73., 217.],
        [217., 361.],
        [361., 505.],
        [  0.,  72.],
        [ 72., 144.],
        [144., 216.],
        [216., 288.],
        [288., 360.],
        [360., 432.],
        [432., 504.],
        [504., 577.],
        [ 37., 109.],
        [109., 181.],
        [181., 253.],
        [253., 325.],
        [325., 397.],
        [397., 469.],
        [469., 541.],
        [  0.,  36.],
        [ 36.,  72.],
        [ 72., 108.],
        [108., 144.],
        [144., 180.],
        [180., 216.],
        [216., 252.],
        [252., 288.],
        [288., 324.],
        [324., 360.],
        [360., 396.],
        [396., 432.],
        [432., 468.],
        [468., 504.],
        [504., 540.],
        [540., 577.],
        [ 19.,  55.],
        [ 55.,  91.],
        [ 91., 127.],
        [1

## Extract Quantiles

In [24]:
def extract_quantiles(X_slice, div=4):
    size = X_slice.shape[-1]
    if size == 1:
        return X_slice.flatten(start_dim=1)
    
    print(size)
    
    num_quantiles = 1 + (size - 1) // div
    if num_quantiles == 1:
        return X_slice.median(dim=1).values.flatten(start_dim=1)
    
    print(num_quantiles)
    
    quantile_points = torch.linspace(0, 1, num_quantiles)
    quantiles = torch.quantile(X_slice, quantile_points, dim=-1).permute(1, 2, 0)
    interval_mean = X_slice.mean(dim=-1, keepdim=True)
    quantiles[..., 1::2] -= interval_mean
    
    return quantiles.flatten(start_dim=1)

interval_start, interval_end = intervals[0]
X_slice = X[..., int(interval_start):int(interval_end)]
features = extract_quantiles(X_slice, div=4)
features

# features.shape

577
145


tensor([[-1.6741, -1.6708, -1.6617, -1.6381, -1.6022, -1.5731, -1.5399, -1.5087,
         -1.4756, -1.4394, -1.3988, -1.3367, -1.3004, -1.2449, -1.2158, -1.2033,
         -1.1999, -1.1886, -1.1781, -1.1646, -1.1547, -1.1413, -1.1299, -1.1157,
         -1.0994, -1.0926, -1.0859, -1.0740, -1.0548, -1.0349, -1.0143, -0.9862,
         -0.9568, -0.9447, -0.9364, -0.9134, -0.9065, -0.8943, -0.8876, -0.8736,
         -0.8600, -0.8322, -0.8086, -0.7883, -0.7659, -0.7371, -0.7081, -0.6790,
         -0.6503, -0.6264, -0.6022, -0.5732, -0.5500, -0.5281, -0.4997, -0.4772,
         -0.4562, -0.4276, -0.4005, -0.3668, -0.3515, -0.3360, -0.3066, -0.2784,
         -0.2501, -0.2234, -0.1945, -0.1676, -0.1646, -0.1359, -0.1045, -0.0783,
         -0.0682, -0.0442, -0.0195,  0.0087,  0.0378,  0.0670,  0.0941,  0.1243,
          0.1517,  0.1811,  0.2125,  0.2410,  0.2672,  0.2976,  0.3291,  0.3581,
          0.3791,  0.3922,  0.4183,  0.4398,  0.4520,  0.4693,  0.4901,  0.5091,
          0.5310,  0.5499,  

# Dataset Upload and Dowload

In [ ]:
# UPLOAD
# from huggingface_hub import upload_folder

# # Push your dataset files
# upload_folder(
#     folder_path="./dataset", repo_id="GowthamNats/NeraThudarDS", repo_type="dataset"
# )

# DOWNLOAD
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="NeraThudar/NeraThudarDS",
#     repo_type="dataset",
#     local_dir="data",
# )
